# DeepCare 03: Baseline models

Trains and evaluates three classical baselines on the splits from `02_preprocessing.ipynb`:

| Model | Input | Imbalance handling |
|---|---|---|
| Logistic Regression | scaled features | `class_weight='balanced'` |
| Random Forest | raw features | `class_weight='balanced_subsample'` |
| XGBoost | raw features | `scale_pos_weight` = negatives / positives |

**Protocol (the Keras MLP in notebook 04 follows the same one)**
- Hyperparameters are chosen by random search, scored by **PR-AUC on the validation set**.
- The decision threshold is chosen on the **validation set** (maximizing F1) and then applied unchanged to the test set.
- The **test set is touched once**, for the final numbers.
- Validation and test probabilities are saved so every model, including the MLP, is compared on identical data.

Note: class weights improve recall but distort probability calibration. Calibration is handled in the evaluation notebook.

## 1. Setup and data

In [1]:
import os, json, joblib, time
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve,
                             confusion_matrix, f1_score)
import xgboost as xgb
from google.colab import drive
drive.mount('/content/drive')

SEED = 42
N_TRIALS = 10            # random-search trials for RF and XGBoost (lower this if runtime is too long)
PROJECT_DIR = '/content/drive/MyDrive/deepcare'
TARGET = 'Diabetes_binary'
D, R = f'{PROJECT_DIR}/data', f'{PROJECT_DIR}/results'
os.makedirs(R, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)

def load(split, kind):
    df = pd.read_csv(f'{D}/{split}_{kind}.csv')
    return df.drop(columns=[TARGET]), df[TARGET].to_numpy()

Xtr_s, ytr = load('train', 'scaled'); Xva_s, yva = load('val', 'scaled'); Xte_s, yte = load('test', 'scaled')
Xtr_r, ytr_r = load('train', 'raw');  Xva_r, yva_r = load('val', 'raw');  Xte_r, yte_r = load('test', 'raw')

# scaled and raw files must be row-aligned
assert np.array_equal(ytr, ytr_r) and np.array_equal(yva, yva_r) and np.array_equal(yte, yte_r)
assert list(Xtr_s.columns) == list(Xtr_r.columns)

print('train/val/test:', len(ytr), len(yva), len(yte))
print('positive rate :', round(ytr.mean(), 4), round(yva.mean(), 4), round(yte.mean(), 4))
print('no-skill PR-AUC (= test prevalence):', round(yte.mean(), 4))

Mounted at /content/drive
train/val/test: 160631 34421 34422
positive rate : 0.1529 0.1529 0.153
no-skill PR-AUC (= test prevalence): 0.153


## 2. Evaluation helpers

Metrics reported at two thresholds: the default 0.5 and the **validation-tuned** threshold. Recall is the metric that matters most for screening, because a missed case costs more than a false alarm.

In [2]:
def best_threshold(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    return float(thr[np.argmax(f1)])

def metrics_at(y, p, t):
    pred = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {'threshold': round(float(t), 4),
            'recall': tp / (tp + fn),
            'precision': tp / (tp + fp) if (tp + fp) else 0.0,
            'specificity': tn / (tn + fp),
            'f1': f1_score(y, pred),
            'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}

results, all_params = [], {}

def evaluate(name, p_val, p_test, params):
    t_star = best_threshold(yva, p_val)
    row = {'model': name,
           'val_pr_auc': average_precision_score(yva, p_val),
           'roc_auc': roc_auc_score(yte, p_test),
           'pr_auc': average_precision_score(yte, p_test)}
    for tag, t in [('@0.5', 0.5), ('@tuned', t_star)]:
        for k, v in metrics_at(yte, p_test, t).items():
            row[f'{k}{tag}'] = v
    results.append(row)
    all_params[name] = params
    np.savez(f'{R}/probs_{name}.npz', val=p_val, test=p_test, val_y=yva, test_y=yte)
    return row

def random_search(name, fit_fn, space, n_trials):
    best = {'score': -1}
    for i, params in enumerate(ParameterSampler(space, n_trials, random_state=SEED)):
        t0 = time.time()
        model, p_val = fit_fn(params)
        score = average_precision_score(yva, p_val)
        print(f'[{name}] trial {i+1}/{n_trials}  val PR-AUC={score:.4f}  ({time.time()-t0:.0f}s)  {params}')
        if score > best['score']:
            best = {'score': score, 'params': params, 'model': model, 'p_val': p_val}
    return best

## 3. Logistic Regression

In [3]:
def fit_lr(params):
    m = LogisticRegression(**params, class_weight='balanced', max_iter=2000, random_state=SEED)
    m.fit(Xtr_s, ytr)
    return m, m.predict_proba(Xva_s)[:, 1]

best_lr = random_search('logreg', fit_lr, {'C': [0.01, 0.1, 1.0, 10.0]}, 4)   # one hyperparameter, so a small grid
lr = best_lr['model']
row = evaluate('logreg', best_lr['p_val'], lr.predict_proba(Xte_s)[:, 1], best_lr['params'])
joblib.dump(lr, f'{PROJECT_DIR}/models/logreg.joblib')
print({k: round(float(v), 4) if isinstance(v, (float, np.floating)) else v for k, v in row.items() if k in ['model','roc_auc','pr_auc','recall@tuned','precision@tuned']})

[logreg] trial 1/4  val PR-AUC=0.4280  (1s)  {'C': 0.01}
[logreg] trial 2/4  val PR-AUC=0.4282  (1s)  {'C': 0.1}
[logreg] trial 3/4  val PR-AUC=0.4282  (1s)  {'C': 1.0}
[logreg] trial 4/4  val PR-AUC=0.4282  (1s)  {'C': 10.0}
{'model': 'logreg', 'roc_auc': 0.8085, 'pr_auc': 0.4147, 'recall@tuned': 0.6387, 'precision@tuned': 0.36}


## 4. Random Forest

Tree depth and leaf size are constrained to keep training time reasonable on ~160k rows. Expect this cell to be the slowest; reduce `N_TRIALS` if needed.

In [4]:
rf_space = {'n_estimators': [150, 250],
            'max_depth': [8, 12, 16],
            'min_samples_leaf': [10, 25, 50, 100],
            'max_features': ['sqrt', 0.5]}

def fit_rf(params):
    m = RandomForestClassifier(**params, class_weight='balanced_subsample', n_jobs=-1, random_state=SEED)
    m.fit(Xtr_r, ytr)
    return m, m.predict_proba(Xva_r)[:, 1]

best_rf = random_search('random_forest', fit_rf, rf_space, N_TRIALS)
rf = best_rf['model']
row = evaluate('random_forest', best_rf['p_val'], rf.predict_proba(Xte_r)[:, 1], best_rf['params'])
print({k: round(float(v), 4) if isinstance(v, (float, np.floating)) else v for k, v in row.items() if k in ['model','roc_auc','pr_auc','recall@tuned','precision@tuned']})
# the fitted forest is large, so it is not saved to Drive; only its probabilities and parameters are kept

[random_forest] trial 1/10  val PR-AUC=0.4413  (43s)  {'n_estimators': 250, 'min_samples_leaf': 25, 'max_features': 0.5, 'max_depth': 12}
[random_forest] trial 2/10  val PR-AUC=0.4331  (29s)  {'n_estimators': 150, 'min_samples_leaf': 10, 'max_features': 0.5, 'max_depth': 16}
[random_forest] trial 3/10  val PR-AUC=0.4413  (25s)  {'n_estimators': 150, 'min_samples_leaf': 25, 'max_features': 0.5, 'max_depth': 12}
[random_forest] trial 4/10  val PR-AUC=0.4385  (45s)  {'n_estimators': 250, 'min_samples_leaf': 25, 'max_features': 0.5, 'max_depth': 16}
[random_forest] trial 5/10  val PR-AUC=0.4388  (25s)  {'n_estimators': 150, 'min_samples_leaf': 10, 'max_features': 0.5, 'max_depth': 12}
[random_forest] trial 6/10  val PR-AUC=0.4439  (26s)  {'n_estimators': 250, 'min_samples_leaf': 50, 'max_features': 'sqrt', 'max_depth': 16}
[random_forest] trial 7/10  val PR-AUC=0.4378  (19s)  {'n_estimators': 150, 'min_samples_leaf': 50, 'max_features': 0.5, 'max_depth': 8}
[random_forest] trial 8/10  val 

## 5. XGBoost

Early stopping uses the validation set, so validation scores are slightly optimistic. The test set is unaffected.

In [5]:
spw = float((ytr == 0).sum() / (ytr == 1).sum())
print('scale_pos_weight:', round(spw, 3))

xgb_space = {'max_depth': [3, 4, 5, 6, 8],
             'learning_rate': [0.03, 0.05, 0.1],
             'subsample': [0.7, 0.85, 1.0],
             'colsample_bytree': [0.6, 0.8, 1.0],
             'min_child_weight': [1, 5, 10],
             'reg_lambda': [1, 5, 10]}

def fit_xgb(params):
    m = xgb.XGBClassifier(**params, n_estimators=2000, tree_method='hist', eval_metric='aucpr',
                          early_stopping_rounds=50, scale_pos_weight=spw,
                          n_jobs=-1, random_state=SEED)
    m.fit(Xtr_r, ytr, eval_set=[(Xva_r, yva)], verbose=False)
    return m, m.predict_proba(Xva_r)[:, 1]

best_xgb = random_search('xgboost', fit_xgb, xgb_space, N_TRIALS)
xg = best_xgb['model']
params = dict(best_xgb['params']); params['best_iteration'] = int(xg.best_iteration)
row = evaluate('xgboost', best_xgb['p_val'], xg.predict_proba(Xte_r)[:, 1], params)
xg.save_model(f'{PROJECT_DIR}/models/xgboost.json')
print({k: round(float(v), 4) if isinstance(v, (float, np.floating)) else v for k, v in row.items() if k in ['model','roc_auc','pr_auc','recall@tuned','precision@tuned']})

scale_pos_weight: 5.538
[xgboost] trial 1/10  val PR-AUC=0.4477  (4s)  {'subsample': 0.85, 'reg_lambda': 1, 'min_child_weight': 10, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
[xgboost] trial 2/10  val PR-AUC=0.4475  (7s)  {'subsample': 1.0, 'reg_lambda': 5, 'min_child_weight': 10, 'max_depth': 4, 'learning_rate': 0.03, 'colsample_bytree': 1.0}
[xgboost] trial 3/10  val PR-AUC=0.4472  (4s)  {'subsample': 1.0, 'reg_lambda': 5, 'min_child_weight': 10, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
[xgboost] trial 4/10  val PR-AUC=0.4495  (5s)  {'subsample': 0.7, 'reg_lambda': 10, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
[xgboost] trial 5/10  val PR-AUC=0.4467  (5s)  {'subsample': 0.7, 'reg_lambda': 1, 'min_child_weight': 10, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
[xgboost] trial 6/10  val PR-AUC=0.4482  (6s)  {'subsample': 0.85, 'reg_lambda': 5, 'min_child_weight': 5, 'max_depth': 8, 

## 6. Results table and saving

In [6]:
res = pd.DataFrame(results).set_index('model')
cols = ['roc_auc', 'pr_auc', 'recall@0.5', 'precision@0.5', 'threshold@tuned',
        'recall@tuned', 'precision@tuned', 'specificity@tuned', 'f1@tuned']
display(res[cols].round(4))

res.to_csv(f'{R}/baseline_results.csv')
with open(f'{R}/baseline_params.json', 'w') as f:
    json.dump(all_params, f, indent=2, default=str)
print('saved results, parameters and probabilities to', R)

,roc_auc,pr_auc,recall@0.5,precision@0.5,threshold@tuned,recall@tuned,precision@tuned,specificity@tuned,f1@tuned
model,,,,,,,,,
logreg,0.8085,0.4147,0.7605,0.3183,0.5962,0.6387,0.3600,0.7949,0.4605
random_forest,0.8150,0.4348,0.7700,0.3204,0.5934,0.6372,0.3746,0.8079,0.4718
xgboost,0.8177,0.4409,0.7956,0.3160,0.6341,0.6184,0.3830,0.8201,0.4730


saved results, parameters and probabilities to /content/drive/MyDrive/deepcare/results
